In [162]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import statsmodels.formula.api as smf

import sys; sys.path.insert(0, '..')
from src.palette import register, content_colors, colorway, CONTENT_ORDER

CONTENT_COLORS = register('light')

In [163]:
conn = sqlite3.connect('../data/lafc_content.db')

In [164]:
with open('../sql/videos_vs_lafc_match_context.sql') as f:
    query = f.read()

df = pd.read_sql(query, conn)
df = df[df['published_at'] >= '2025-01-01T00:00:00Z'].copy()
df['content_type'] = df['content_type'].fillna('no_playlist')
df['playlist'] = df['playlist'].fillna('(no playlist)')
df

,video_id,title,description,published_at,duration,view_count,like_count,comment_count,engagement_rate,format,...,goals_for,goals_against,lafc_points,lafc_played,lafc_wins,opp_points,opp_played,opp_wins,days_since_match,days_until_match
0,IxrFLgowFd4,Armindo Sieb is Black & Gold.,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T16:40:02Z,PT52S,275,23,7,0.10909,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.72,NaN
1,HugEGKBw0kk,LAFC vs QRO | Postmatch Media,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T09:02:51Z,PT14M19S,306,22,26,0.15686,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.40,NaN
2,pLVoxNTyGLI,The top scorer in Leagues Cup history 📈,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:30:24Z,PT15S,6287,169,12,0.02879,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.33,NaN
3,wz6UrdGQWjY,BOUANGA EQUALIZER 💥,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:15:01Z,PT13S,3940,91,4,0.02411,short,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.32,NaN
4,SqgJkPzCN6Y,Denis Bouanga equalizes against Querétaro,Watch LAFC on *MLS Season Pass* on the  Apple...,2026-08-13T07:02:06Z,PT13S,1712,55,6,0.03563,horizontal,...,1.0,1.0,33.0,18.0,10.0,33.0,16.0,10.0,11.31,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,zPYzE9H0Zyo,Igor Jesus is Black & Gold,📝 #LAFC acquires Igor Jesus from Portuguese Pr...,2025-01-21T22:36:06Z,PT1M24S,1221,43,5,0.03931,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,93.90,31.96
1211,Uycrl281Zjg,"Part of our History | Thank you, Erik Dueñas",The LAFC Original. Forever Black & Gold.\n\nBe...,2025-01-14T19:59:43Z,PT1M30S,1518,43,6,0.03228,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,86.79,39.07
1212,xkQPq_y1QOg,And now for some midfield thunder 🔨⚡,Odin Thiago Holm is Balck & Gold.\n\nWatch LAF...,2025-01-13T19:44:09Z,PT19S,2473,143,12,0.06268,short,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,85.78,40.08
1213,bm7P0aOOhAo,Odin Thiago Holm is Black & Gold,LAFC has acquired Norwegian midfielder Odin Th...,2025-01-13T19:34:11Z,PT1M23S,2684,67,8,0.02794,horizontal,...,3.0,1.0,61.0,33.0,18.0,21.0,33.0,6.0,85.77,40.09


In [165]:
df.shape

(1215, 29)

In [166]:
df.columns

Index(['video_id', 'title', 'description', 'published_at', 'duration',
       'view_count', 'like_count', 'comment_count', 'engagement_rate',
       'format', 'playlist', 'n_playlists', 'all_playlists', 'content_type',
       'season', 'kickoff_utc', 'opponent', 'result', 'home_away', 'goals_for',
       'goals_against', 'lafc_points', 'lafc_played', 'lafc_wins',
       'opp_points', 'opp_played', 'opp_wins', 'days_since_match',
       'days_until_match'],
      dtype='str')

In [167]:
fig = px.histogram(
    df, x='view_count',
    title="Videos per view - linear scale"
    )

fig.update_xaxes(title='Views')

fig.update_yaxes(title='Number of videos')

fig.show()

In [168]:
df['log10_views'] = np.log10(df['view_count'])

fig = px.histogram(
    df, x='log10_views',
    nbins=80,
    title='Videos per View - Log Scale', 
    )
fig.update_xaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'],
    title='Views')

fig.update_yaxes(
    title='Number of videos'
)

fig.show()

In [169]:
fig = px.histogram(
    df, x='engagement_rate',
    nbins=80,
    title= 'Engagement rate per video'
    )

fig.update_yaxes()

fig.update_xaxes(tickformat='.1%')

fig.show()



In [170]:
df['format'].value_counts()

format
horizontal    741
short         398
live           76
Name: count, dtype: int64

In [171]:
df.groupby('format')['view_count'].median().sort_values(ascending=False)

format
short         8331.0
live          2023.0
horizontal    1700.0
Name: view_count, dtype: float64

In [172]:
df['playback_type'] = np.where(df['format'] == 'short', 'short', 'horizontal+live')

order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='view_count',
    log_y=True,
    color='playback_type',
    category_orders={'playback_type': order},
    title='View Count by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'view_count': 'View Count'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [173]:
order = ['short', 'horizontal+live']
n = df['playback_type'].value_counts()

fig = px.box(
    df, x='playback_type', y='engagement_rate',
    color='playback_type',
    category_orders={'playback_type': order},
    title='Engagement Rate by Playback Type - Short vs. Horizontal+Live',
    labels={'playback_type': '', 'engagement_rate': 'Engagement Rate'},
)

fig.update_traces(marker=dict(opacity=0.3, size=4), jitter=0.4)

fig.update_layout(showlegend=False)

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [174]:
table = pd.crosstab(df['format'], df['content_type'])
table

content_type,community,feature,full_match,highlights,match_preview,no_playlist,podcast,press_interview,show,unclassified
format,,,,,,,,,,
horizontal,1,5,6,171,30,57,184,155,82,50
live,0,0,2,0,0,3,69,0,0,2
short,0,0,0,46,1,322,0,0,0,29


In [175]:
shorts_df = df[df['format'] == 'short'].copy()
playlist_table = shorts_df['playlist'].value_counts(dropna=False)
content_type_table = shorts_df['content_type'].value_counts(dropna=False)

display(content_type_table)
display(playlist_table)

content_type
no_playlist      322
highlights        46
unclassified      29
match_preview      1
Name: count, dtype: int64

playlist
(no playlist)        322
Highlights            46
The Son Spotlight     29
Match Previews         1
Name: count, dtype: int64

In [176]:
order = shorts_df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = shorts_df['content_type'].value_counts()

fig = px.box(
    shorts_df, x='content_type', y='log10_views',
    color='content_type',
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type - Short Format',
    labels={'content_type': 'Content Type', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [177]:
order = shorts_df.groupby('content_type')['engagement_rate'].median().sort_values().index.tolist()
n = shorts_df['content_type'].value_counts()

fig = px.box(
    shorts_df, x='content_type', y='engagement_rate',
    color='content_type',
    category_orders={'content_type': order},
    title='Engagement Rate (log 10) by Content Type - Short Format',
    labels={'content_type': 'Content Type', 'engagment_rate': "Engagement Rate"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [178]:
unclassified_shorts_df = shorts_df[shorts_df['content_type'] == 'unclassified'].copy()
unclassified_shorts_df['playlist'].value_counts(dropna=False)

playlist
The Son Spotlight    29
Name: count, dtype: int64

In [179]:
horizontal_df = df[df['format'] != 'short'].copy()

order = horizontal_df.groupby('content_type')['view_count'].median().sort_values().index.tolist()
n = horizontal_df['content_type'].value_counts()

fig = px.box(
    horizontal_df, x='content_type', y='log10_views',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Views (log 10) by Content Type - Horizontal / Live ',
    labels={'content_type': 'Content Type', 'log10_views': "Views"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(
    tickvals=[2, 3, 4, 5, 6],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()

In [180]:
order = horizontal_df.groupby('content_type')['engagement_rate'].median().sort_values().index.tolist()
n = horizontal_df['content_type'].value_counts()

fig = px.box(
    horizontal_df, x='content_type', y='engagement_rate',
    color='content_type',
    color_discrete_map=CONTENT_COLORS,
    category_orders={'content_type': order},
    title='Engagement Rate by Content Type - Horizontal / Live',
    labels={'content_type': 'Content Type', 'engagement_rate': "Engagement Rate"}
    )

fig.update_xaxes(
    tickvals=order,
    ticktext=[f'{f}<br>n={n[f]}' for f in order])

fig.update_yaxes(tickformat='.1%')

fig.show()

In [181]:
unclassified_horizontal_df = horizontal_df[horizontal_df['content_type'] == 'unclassified'].copy()
unclassified_horizontal_df['playlist'].value_counts(dropna=False)

playlist
The Son Spotlight    40
Major News            8
The Vela Vault        4
Name: count, dtype: int64

In [182]:
son_spotlight_df = unclassified_horizontal_df[unclassified_horizontal_df['playlist'] == 'The Son Spotlight'].copy()
son_spotlight_df[['title', 'playlist']]

,title,playlist
20,One Year of Sonny Goals,The Son Spotlight
36,Sonny vs. SKC | EVERY ANGLE,The Son Spotlight
52,SONNY SCORES HIS 3RD GOAL IN 3 MATCHES | LAFC ...,The Son Spotlight
54,Sonny vs RSL | EVERY ANGLE,The Son Spotlight
67,SONNY SCORES FROM THE TOP OF THE BOX | LAFC vs...,The Son Spotlight
79,Son Heung-Min | EVERY ANGLE of his derby goal ...,The Son Spotlight
92,SONNY’S FIRST GOAL OF THE SEASON | LAG vs LAFC,The Son Spotlight
112,Who has the most aura? w/ Sonny | Ryan & Aaron...,The Son Spotlight
120,Sonny bobblehead arrives in Los Angeles | LAFC...,The Son Spotlight
236,Sonny's goal from the stands | LAFC vs Cruz Azul,The Son Spotlight


In [183]:
order = horizontal_df.groupby('playlist')['view_count'].median().sort_values().index.tolist()
n = horizontal_df['playlist'].value_counts()

fig = px.box(
    horizontal_df, y='playlist', x='view_count',
    log_x=True,
    color='playlist',
    category_orders={'playlist': order},
    title='View Count (Log Scale) by Playlist - Horizontal / Live',
    labels={'playlist': 'Playlist', 'view_count': 'View Count'},
    height=600,
)

fig.update_layout(showlegend=False)

fig.update_yaxes(
    tickvals=order,
    ticktext=[f'{p}  (n={n[p]})' for p in order])

fig.update_xaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()


In [184]:
order = df.groupby('playlist')['engagement_rate'].median().sort_values().index.tolist()
n = df['playlist'].value_counts()

fig = px.box(
    df, y='playlist', x='engagement_rate',
    log_x=True,
    color='playlist',
    category_orders={'playlist': order},
    title='Engagement Rate by Playlist - Horizontal / Live',
    labels={'playlist': 'Playlist', 'engagement_rate': 'Engagement Rate'},
    height=600,
)

fig.update_layout(showlegend=False)

fig.update_yaxes(
    tickvals=order,
    ticktext=[f'{p}  (n={n[p]})' for p in order])

fig.update_xaxes(
    tickvals=[100, 1_000, 10_000, 100_000, 1_000_000],
    ticktext=['100', '1K', '10K', '100K', '1M'])

fig.show()


<h2>Timing</h2>

In [185]:
# Assigns the two columns into two series.
after  = df['days_since_match']    # days SINCE the previous match (always ≥ 0)
before = df['days_until_match']    # days UNTIL the next match     (always ≥ 0)


# Fills in na with infinity, then compares the two series, row by row, and returns a boolean. If before is smaller - then it returns true, meaning this row is closer to the NEXT match.
closer_to_next = before.fillna(np.inf) < after.fillna(np.inf)

# Assigns either a -before or after based on the boolean in closer_to_next
df['days_from_match'] = np.where(closer_to_next, -before, after)

#Overwrite nan if its been 21 days since (and until) the closest matches

OFFSEASON_DAYS = 21
not_in_cycle = ((after.fillna(np.inf)  > OFFSEASON_DAYS) &
                (before.fillna(np.inf) > OFFSEASON_DAYS))

df.loc[not_in_cycle, 'days_from_match'] = np.nan

print('not in a cycle :', not_in_cycle.sum())
print('before a match :', (df['days_from_match'] < 0).sum())
print('after a match  :', (df['days_from_match'] > 0).sum())

not in a cycle : 71
before a match : 485
after a match  : 659


In [186]:
#Binning and adding the bin info back to the df.

CYCLE_EDGES  = [-np.inf, -7, -3, -1, 0, 2, 4, 8, np.inf]
CYCLE_LABELS = ['7+ before', '3-7 before', '1-3 before', '0-1 before',
                '0-2 after', '2-4 after', '4-8 after', '8+ after']

df['cycle_bin'] = pd.cut(
    df['days_from_match'],
    bins=CYCLE_EDGES,
    labels=CYCLE_LABELS, right=False
    )

display(df[['title', 'days_since_match', 'days_until_match', 'cycle_bin']])

,title,days_since_match,days_until_match,cycle_bin
0,Armindo Sieb is Black & Gold.,11.72,NaN,8+ after
1,LAFC vs QRO | Postmatch Media,11.40,NaN,8+ after
2,The top scorer in Leagues Cup history 📈,11.33,NaN,8+ after
3,BOUANGA EQUALIZER 💥,11.32,NaN,8+ after
4,Denis Bouanga equalizes against Querétaro,11.31,NaN,8+ after
...,...,...,...,...
1210,Igor Jesus is Black & Gold,93.90,31.96,NaN
1211,"Part of our History | Thank you, Erik Dueñas",86.79,39.07,NaN
1212,And now for some midfield thunder 🔨⚡,85.78,40.08,NaN
1213,Odin Thiago Holm is Black & Gold,85.77,40.09,NaN


In [187]:
print(df['cycle_bin'].value_counts(dropna=False).sort_index())

cycle_bin
7+ before      50
3-7 before     97
1-3 before    223
0-1 before    115
0-2 after     423
2-4 after     117
4-8 after      65
8+ after       54
NaN            71
Name: count, dtype: int64


In [188]:
PLOT_BINS = CYCLE_LABELS[1:-1]      # drop '7+ before' and '8+ after'

plot_df = df[df['cycle_bin'].isin(PLOT_BINS)].dropna(subset=['engagement_rate']).copy()
plot_df['cycle_bin'] = plot_df['cycle_bin'].cat.remove_unused_categories()
n = plot_df['cycle_bin'].value_counts()

In [189]:
fig = px.box(
    plot_df, x='cycle_bin', y='view_count',
    log_y=True,
    category_orders={'cycle_bin': PLOT_BINS},
    color='cycle_bin',
    points=False,
    title='View Count Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'view_count': 'View Count (log)'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.add_vline(x=2.5, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [190]:
fig = px.box(
    plot_df, x='cycle_bin', y='engagement_rate',
    color='cycle_bin',
    category_orders={'cycle_bin': PLOT_BINS},
    points=False,
    title='Engagement Rate Across the Match Cycle',
    labels={'cycle_bin': 'Position in match cycle',
            'engagement_rate': 'Engagement Rate'},
)

fig.update_xaxes(
    tickvals=PLOT_BINS,
    ticktext=[f'{b}<br>n={n[b]}' for b in PLOT_BINS])

fig.update_yaxes(tickformat='.1%')

fig.add_vline(x=2.5, line_dash='dot', line_color='#888888',
              annotation_text='kickoff', annotation_position='top')

fig.show()

In [191]:
table = plot_df.groupby('cycle_bin', observed=True).agg(
        n=('view_count', 'size'),
        median_views=('view_count', 'median'),
        median_engagement_rate=('engagement_rate', 'median'))

table['median_engagement_rate'] = (table['median_engagement_rate'] * 100).round(2).astype(str) + '%'

table

,n,median_views,median_engagement_rate
cycle_bin,,,
3-7 before,97,2869.0,4.81%
1-3 before,223,2266.0,5.82%
0-1 before,115,2180.0,5.98%
0-2 after,423,4461.0,4.29%
2-4 after,117,1827.0,5.41%
4-8 after,65,3087.0,4.66%


<h2>Regression</h2>

In [192]:
df['is_short'] = (df['format'] == 'short')
df['is_son'] = (df['playlist'] == 'The Son Spotlight')
df['is_matchday'] = df['days_from_match'].between(-1, 0)

model = smf.ols('log10_views ~ is_short + is_son + is_matchday + C(content_type)',
            data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            log10_views   R-squared:                       0.424
Model:                            OLS   Adj. R-squared:                  0.419
Method:                 Least Squares   F-statistic:                     73.88
Date:                Tue, 18 Aug 2026   Prob (F-statistic):          5.86e-135
Time:                        18:55:57   Log-Likelihood:                -1093.5
No. Observations:                1215   AIC:                             2213.
Df Residuals:                    1202   BIC:                             2279.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte